# New Workflow Component for Expanding Knowledge

This document introduces a proposed workflow enhancement, with **binning** as a case study to illustrate the justification for dataset reprocessing. Specifically, **Eukaryotic binning**, operational since **September 2022**, serves as an example.

## Summary of EukCC Bin Statistics and Taxonomic Distribution  

As of **January 23, 2025**, the following statistics have been compiled:

- **[Total Metagenomes with Bins: 25,146](https://img.jgi.doe.gov/cgi-bin/m/main.cgi?section=MetagenomeBins&page=bins&type=taxonomy)**  
- **Total Bins: 319,839**  
- **Total Metagenomes with EukCC Bins: 827**  
- **[Total EukCC Bins: 1,281](https://img.jgi.doe.gov/cgi-bin/m/main.cgi?section=MetagenomeBins&page=bindetail&taxonomy=ncbi&type=bytaxonomy&domain=Eukaryota&domain=Eukaryota%20EukCC%20Bins)**  

### Additional Notes  
The total number of EukCC bins has been exported as a TSV file (`data/exported_img_data-EukCC_bins.tsv`) using the IMG database link provided above. This file serves as a critical reference for downstream analyses.

---

## Analysis: EukCC Bin Counts per Metagenome  

An initial analysis was performed to evaluate the distribution of EukCC bins across metagenomes:  

- A histogram of EukCC bin counts per metagenome reveals that **most metagenomes contain a single EukCC bin**, with the maximum number of bins in a single metagenome being **9**.

---

## Analysis: EukCC Bin Taxonomy Distribution  

The taxonomy distribution analysis highlights a dominance of Eukaryota lineages. Key findings include:

- Significant representation of **Chlorophyta**, particularly within the **Mamiellophyceae** class and related orders.
- Prominent presence of **Ascomycota**, with notable classes such as **Lecanoromycetes**.  
- Recurring observations of fungal lineages such as **Basidiomycota** and **Tremellomycetes**.  

A **Sankey diagram** has been generated to visualize hierarchical taxonomic connections. The diagram illustrates the flow from higher-order taxa (e.g., **Eukaryota**) to specific families and genera, emphasizing dominant relationships and abundances within the dataset. 

A **Krona plot** provided another perspective to view the taxnomy distribution which is run through KronaTools v2.8.1 and can be generated by following command lines:

```js
awk -F"\t" '{print $5}' data/exported_img_data-EukCC_bins.txt  | sed -e 's/; /\t/g' | tail -n+2 > exported_img_data-EukCC_bins_for_krona.txt
ktImportText -q -o data/EukCC.krona.html exported_img_data-EukCC_bins_for_krona.txt 

```

---

## Conclusion  

This analysis demonstrates the value of reprocessing datasets with the proposed workflow component. By systematically updating processing methodologies, we can ensure more accurate and comprehensive insights into eukaryotic binning trends, enabling improved knowledge discovery and resource utilization.


In [12]:
import pandas as pd
import plotly.express as px

# Load the data
file_path = 'data/exported_img_data-EukCC_bins.tsv'
data = pd.read_csv(file_path, sep='\t')

total_EukCC_bins = len(data['IMG Genome ID'])
total_mtg_with_EukCC_bins = len(data['IMG Genome ID'].unique())

fig = px.histogram(
        data.groupby('IMG Genome ID')['bin_oid'].count(),
        title=f'EukCC_Bin_Count_by_Metagenomes<br>  Total Metagenomes with EukCC Bins: {total_mtg_with_EukCC_bins} <br>  Total EukCC Bins: {total_EukCC_bins}',
    )
fig.update_layout(xaxis_title='EukCC Bin Count', yaxis_title='Metagenomes',showlegend=False)
fig.show()

In [11]:
import plotly.graph_objects as go

# Extract relevant columns for Sankey diagram from the above data
lineages = data['Bin Lineage'].dropna()
# Split lineages into hierarchical components
hierarchies = lineages.str.split(';')

# Create unique node labels
node_labels = list(set([level for lineage in hierarchies for level in lineage]))
node_indices = {label: idx for idx, label in enumerate(node_labels)}

# Generate Sankey diagram links
source = []
target = []
values = []

for lineage in hierarchies:
    for i in range(len(lineage) - 1):
        source.append(node_indices[lineage[i]])
        target.append(node_indices[lineage[i + 1]])
        values.append(1)  # Each occurrence adds a count

# Create the Sankey diagram
fig = go.Figure(go.Sankey(
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=node_labels
    ),
    link=dict(
        source=source,
        target=target,
        value=values,
        color="rgba(128,128,128, 0.2)",
        hovercolor="midnightblue"
    )
))

fig.update_layout(title_text=f"Sankey Diagram of EukCC Bins Lineage<br>  Total Metagenomes with EukCC Bins: {total_mtg_with_EukCC_bins} <br>  Total EukCC Bins: {total_EukCC_bins}", font_size=10, width=800,
    height=600, margin=dict(l=50, r=50, t=50, b=50))
fig.show()